# DPO: Direct Preference Optimization - Simplified RLHF

**Goal**: Learn how DPO simplifies RLHF by directly optimizing from preference data without needing a separate reward model or complex RL.

**What you'll learn:**
- Why DPO is simpler than PPO-based RLHF
- How to train models with DPO
- The mathematical intuition behind DPO
- When to use DPO vs traditional RLHF
- Practical tips for hyperparameter tuning

**Prerequisites:** Understanding of SFT (notebook 01) and Reward Modeling (notebook 02)

---

## Part 1: DPO vs Traditional RLHF

### Traditional RLHF Pipeline (3 Stages)

```
┌─────────────────────────────────────────────────┐
│  Stage 1: Supervised Fine-Tuning (SFT)         │
│  Train on demonstrations → SFT Model           │
└─────────────────────────────────────────────────┘
                    ↓
┌─────────────────────────────────────────────────┐
│  Stage 2: Reward Modeling                       │
│  Train on preferences → Reward Model            │
└─────────────────────────────────────────────────┘
                    ↓
┌─────────────────────────────────────────────────┐
│  Stage 3: PPO Training (COMPLEX!)               │
│  - 4 models: Policy, Critic, Reference, Reward │
│  - Generate rollouts                            │
│  - Compute advantages                           │
│  - Update with clipped objective                │
└─────────────────────────────────────────────────┘
```

**Problems:**
- 🔴 Complex: 3 stages, 4 models
- 🔴 Unstable: RL can diverge
- 🔴 Slow: Many rollouts and updates
- 🔴 Sensitive: Many hyperparameters

### DPO Pipeline (2 Stages)

```
┌─────────────────────────────────────────────────┐
│  Stage 1: Supervised Fine-Tuning (SFT)         │
│  Train on demonstrations → SFT Model           │
└─────────────────────────────────────────────────┘
                    ↓
┌─────────────────────────────────────────────────┐
│  Stage 2: DPO Training (SIMPLE!)                │
│  - 2 models: Policy (trainable), Reference     │
│  - Direct supervised learning                   │
│  - No reward model needed                       │
│  - No RL complexity                             │
└─────────────────────────────────────────────────┘
```

**Advantages:**
- ✅ Simpler: 2 stages, 2 models
- ✅ Stable: Supervised learning
- ✅ Fast: Direct optimization
- ✅ Easy to tune: Few hyperparameters
- ✅ Same or better results!

**Key Insight:** DPO reformulates RLHF as a supervised learning problem!

## Part 2: Setup and Imports

In [ ]:
# Standard imports
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import TrainingArguments, set_seed

# Our modules
from src.models.language import LanguageModel
from src.core.dpo.trainer import DPOTrainer, compute_dpo_metrics
from src.core.dpo.loss import dpo_loss, ipo_loss, compute_sequence_log_probs
from src.data.processors.preference import (
    create_synthetic_preference_data,
    prepare_preference_data,
    PreferenceDataCollator,
)

# Set random seed for reproducibility
set_seed(42)

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## Part 3: Understanding DPO Math (Interactive)

### The DPO Loss Function

**DPO Loss:**
$$L_{\text{DPO}} = -\log \sigma\left(\beta \cdot \left[\log\frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \log\frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right]\right)$$

Where:
- $\pi_\theta$ = policy model (being trained)
- $\pi_{\text{ref}}$ = reference model (frozen SFT model)
- $y_w$ = chosen/winner response
- $y_l$ = rejected/loser response
- $\beta$ = temperature controlling KL penalty
- $\sigma$ = sigmoid function

**Intuition:**
1. Compute how much policy prefers chosen vs rejected (relative to reference)
2. Scale by $\beta$ (controls how aggressively we optimize)
3. Push through sigmoid and take negative log (classification loss)

**Implicit Reward:**
$$r(x, y) = \beta \cdot \log\frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)}$$

This is the "reward" the model assigns to response $y$!

In [ ]:
# Visualize how DPO loss works
def visualize_dpo_loss():
    """Visualize DPO loss as a function of log ratio differences."""
    # Simulate log ratios
    log_ratio_diff = np.linspace(-5, 5, 100)
    beta = 0.1
    
    # Compute DPO loss for each difference
    logits = beta * log_ratio_diff
    loss = -np.log(1 / (1 + np.exp(-logits)))  # -log(sigmoid(x))
    
    # Plot
    plt.figure(figsize=(12, 5))
    
    # Left: Loss curve
    plt.subplot(1, 2, 1)
    plt.plot(log_ratio_diff, loss, linewidth=2)
    plt.axvline(x=0, color='r', linestyle='--', label='Equal preference')
    plt.xlabel('log(π/π_ref)[chosen] - log(π/π_ref)[rejected]')
    plt.ylabel('DPO Loss')
    plt.title('DPO Loss Curve')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Add annotations
    plt.annotate('Chosen strongly\npreferred\n(low loss)',
                xy=(3, 0.1), fontsize=10, ha='center')
    plt.annotate('Rejected strongly\npreferred\n(high loss)',
                xy=(-3, 4), fontsize=10, ha='center')
    
    # Right: Gradient magnitude
    plt.subplot(1, 2, 2)
    gradient_magnitude = np.abs(beta / (1 + np.exp(logits)))
    plt.plot(log_ratio_diff, gradient_magnitude, linewidth=2, color='orange')
    plt.xlabel('log(π/π_ref)[chosen] - log(π/π_ref)[rejected]')
    plt.ylabel('|Gradient|')
    plt.title('Gradient Magnitude')
    plt.grid(True, alpha=0.3)
    
    plt.annotate('Strongest learning\nsignal here',
                xy=(0, 0.025), fontsize=10, ha='center')
    
    plt.tight_layout()
    plt.show()
    
    print("📊 Interpretation:")
    print("  Left: Loss is lowest when policy strongly prefers chosen over rejected")
    print("  Right: Gradient is strongest when policy is uncertain (near 0)")
    print("        → Model learns most from examples where it's confused!")

visualize_dpo_loss()

## Part 4: Create Synthetic Preference Data

Let's create some synthetic preference pairs to understand the data format.

In [ ]:
# Generate synthetic preference data
print("Generating synthetic preference data...")
train_examples = create_synthetic_preference_data(num_examples=100, seed=42)
eval_examples = create_synthetic_preference_data(num_examples=20, seed=43)

print(f"\nGenerated {len(train_examples)} training examples")
print(f"Generated {len(eval_examples)} evaluation examples")

# Show example
example = train_examples[0]
print("\n" + "="*80)
print("EXAMPLE PREFERENCE PAIR")
print("="*80)
print(f"\n📝 Prompt:\n{example['prompt']}")
print(f"\n✅ Chosen (preferred response):\n{example['chosen']}")
print(f"\n❌ Rejected (worse response):\n{example['rejected']}")
print("\n" + "="*80)

print("\n💡 DPO will learn to prefer chosen over rejected!")

## Part 5: Load and Prepare Models

DPO requires two models:
1. **Policy model** (trainable): Starts from SFT, will be optimized
2. **Reference model** (frozen): Copy of SFT, provides KL constraint

In [ ]:
print("Loading models...\n")

# Load policy model (will be trained)
print("1️⃣ Loading policy model (trainable)...")
policy_model = LanguageModel.from_pretrained(
    "gpt2",
    use_lora=True,  # Use LoRA for efficiency
    lora_config={
        "r": 8,
        "lora_alpha": 16,
        "target_modules": ["c_attn"],
        "lora_dropout": 0.05,
    },
    device=device,
)

print(f"   Total parameters: {policy_model.num_parameters:,}")
print(f"   Trainable parameters: {policy_model.num_trainable_parameters:,}")
print(f"   Trainable %: {policy_model.num_trainable_parameters / policy_model.num_parameters * 100:.2f}%")

# Load reference model (frozen)
print("\n2️⃣ Loading reference model (frozen)...")
reference_model = LanguageModel.from_pretrained(
    "gpt2",
    use_lora=False,  # No LoRA for reference
    device=device,
)

print(f"   Total parameters: {reference_model.num_parameters:,}")
print(f"   All parameters frozen ❄️")

print("\n✅ Models loaded successfully!")
print("\n💡 Policy will learn from preferences, reference provides stability")

## Part 6: Manual DPO Step-by-Step

Before using the trainer, let's manually compute DPO loss for one example to understand what's happening.

In [ ]:
# Prepare one example
tokenizer = policy_model.tokenizer

example = train_examples[0]
chosen_text = example['prompt'] + example['chosen']
rejected_text = example['prompt'] + example['rejected']

# Tokenize
chosen_tokens = tokenizer(chosen_text, return_tensors="pt", truncation=True, max_length=128)
rejected_tokens = tokenizer(rejected_text, return_tensors="pt", truncation=True, max_length=128)

# Move to device
chosen_tokens = {k: v.to(device) for k, v in chosen_tokens.items()}
rejected_tokens = {k: v.to(device) for k, v in rejected_tokens.items()}

print("Step-by-step DPO computation:\n" + "="*80)

# Step 1: Forward pass through policy
print("\n1️⃣ Forward pass through policy model...")
with torch.no_grad():
    policy_chosen_output = policy_model.model(**chosen_tokens)
    policy_rejected_output = policy_model.model(**rejected_tokens)

# Step 2: Forward pass through reference
print("2️⃣ Forward pass through reference model...")
with torch.no_grad():
    ref_chosen_output = reference_model.model(**chosen_tokens)
    ref_rejected_output = reference_model.model(**rejected_tokens)

# Step 3: Compute log probabilities
print("3️⃣ Computing log probabilities...")
policy_chosen_logp = compute_sequence_log_probs(
    policy_chosen_output.logits,
    chosen_tokens['input_ids'],
    chosen_tokens['attention_mask']
)
policy_rejected_logp = compute_sequence_log_probs(
    policy_rejected_output.logits,
    rejected_tokens['input_ids'],
    rejected_tokens['attention_mask']
)
ref_chosen_logp = compute_sequence_log_probs(
    ref_chosen_output.logits,
    chosen_tokens['input_ids'],
    chosen_tokens['attention_mask']
)
ref_rejected_logp = compute_sequence_log_probs(
    ref_rejected_output.logits,
    rejected_tokens['input_ids'],
    rejected_tokens['attention_mask']
)

print(f"   Policy log P(chosen):   {policy_chosen_logp.item():.2f}")
print(f"   Policy log P(rejected): {policy_rejected_logp.item():.2f}")
print(f"   Ref log P(chosen):      {ref_chosen_logp.item():.2f}")
print(f"   Ref log P(rejected):    {ref_rejected_logp.item():.2f}")

# Step 4: Compute log ratios
print("\n4️⃣ Computing log ratios (π/π_ref)...")
log_ratio_chosen = policy_chosen_logp - ref_chosen_logp
log_ratio_rejected = policy_rejected_logp - ref_rejected_logp

print(f"   log(π/π_ref)[chosen]:   {log_ratio_chosen.item():.4f}")
print(f"   log(π/π_ref)[rejected]: {log_ratio_rejected.item():.4f}")

# Step 5: Compute implicit rewards
print("\n5️⃣ Computing implicit rewards...")
beta = 0.1
reward_chosen = beta * log_ratio_chosen
reward_rejected = beta * log_ratio_rejected
reward_margin = reward_chosen - reward_rejected

print(f"   Reward(chosen):   {reward_chosen.item():.4f}")
print(f"   Reward(rejected): {reward_rejected.item():.4f}")
print(f"   Margin:           {reward_margin.item():.4f}")

if reward_margin > 0:
    print("   ✅ Policy already prefers chosen! (margin > 0)")
else:
    print("   ❌ Policy prefers rejected! (margin < 0) → needs training")

# Step 6: Compute DPO loss
print("\n6️⃣ Computing DPO loss...")
loss, details = dpo_loss(
    policy_chosen_logp,
    policy_rejected_logp,
    ref_chosen_logp,
    ref_rejected_logp,
    beta=beta,
    return_details=True,
)

print(f"   Loss: {loss.item():.4f}")
print(f"   Accuracy: {details['accuracy']:.2%}")
print(f"   KL divergence (chosen): {details['chosen_kl']:.4f}")
print(f"   KL divergence (rejected): {details['rejected_kl']:.4f}")

print("\n" + "="*80)
print("\n💡 Training will minimize this loss, making policy prefer chosen over rejected!")

## Part 7: Train with DPO

Now let's use the DPOTrainer to train on our synthetic data.

In [ ]:
# Prepare datasets
print("Preparing datasets...")
train_data = Dataset.from_list(train_examples)
eval_data = Dataset.from_list(eval_examples)

train_dataset = prepare_preference_data(
    train_data,
    tokenizer,
    max_length=128,
)

eval_dataset = prepare_preference_data(
    eval_data,
    tokenizer,
    max_length=128,
)

print(f"Train dataset: {len(train_dataset)} pairs")
print(f"Eval dataset: {len(eval_dataset)} pairs")

# Create data collator
data_collator = PreferenceDataCollator(
    tokenizer=tokenizer,
    max_length=128,
)

In [ ]:
# Setup training arguments
output_dir = "./outputs/dpo_tutorial"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,  # Effective batch = 8
    learning_rate=5e-7,  # Much lower than SFT!
    weight_decay=0.0,
    warmup_steps=10,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=25,
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy",
    greater_is_better=True,
    fp16=False,
    report_to=[],  # Disable wandb for tutorial
)

print("Training configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Device: {device}")

In [ ]:
# Create DPO trainer
print("\nCreating DPO trainer...")

trainer = DPOTrainer(
    model=policy_model.model,  # Pass inner model
    ref_model=reference_model.model,  # Pass inner model
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_dpo_metrics,
    beta=0.1,  # KL penalty coefficient
    loss_type="dpo",  # Can also use "ipo"
    log_rewards=True,
    num_rewards_to_log=3,
)

print("✅ Trainer created!")
print("\n🚀 Starting training...\n")
print("="*80)

In [ ]:
# Train!
train_result = trainer.train()

print("\n" + "="*80)
print("\n✅ Training complete!")
print(f"\nTraining time: {train_result.metrics['train_runtime']:.2f} seconds")
print(f"Final loss: {train_result.metrics['train_loss']:.4f}")

## Part 8: Evaluate the Trained Model

In [ ]:
# Evaluate on test set
print("Evaluating trained model...\n")
eval_results = trainer.evaluate()

print("="*80)
print("EVALUATION RESULTS")
print("="*80)
print(f"\nAccuracy: {eval_results['eval_accuracy']:.2%}")
print(f"  → Model correctly prefers chosen over rejected {eval_results['eval_accuracy']:.0%} of the time")

print(f"\nReward Margin: {eval_results['eval_margin_mean']:.4f}")
print(f"  → Average difference between chosen and rejected rewards")

print(f"\nChosen Reward: {eval_results['eval_chosen_mean']:.4f} ± {eval_results['eval_chosen_std']:.4f}")
print(f"Rejected Reward: {eval_results['eval_rejected_mean']:.4f} ± {eval_results['eval_rejected_std']:.4f}")

print("\n" + "="*80)

# Interpret results
accuracy = eval_results['eval_accuracy']
if accuracy > 0.70:
    print("\n🎉 Excellent! Model learned preferences well (>70% accuracy)")
elif accuracy > 0.60:
    print("\n👍 Good! Model learned preferences reasonably (>60% accuracy)")
elif accuracy > 0.50:
    print("\n⚠️  Model learned something, but could be better (>50% accuracy)")
else:
    print("\n❌ Model didn't learn effectively (≤50% accuracy = random guessing)")

## Part 9: Visualize Training Progress

In [ ]:
# Get training metrics
metrics = trainer.get_training_metrics()

if metrics['steps']:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: Loss
    axes[0, 0].plot(metrics['steps'], metrics['losses'], linewidth=2)
    axes[0, 0].set_xlabel('Training Steps')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('DPO Loss Over Time')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot 2: Accuracy
    if metrics['accuracies']:
        axes[0, 1].plot(metrics['steps'][:len(metrics['accuracies'])], 
                       metrics['accuracies'], linewidth=2, color='green')
        axes[0, 1].axhline(y=0.5, color='r', linestyle='--', label='Random guessing')
        axes[0, 1].set_xlabel('Training Steps')
        axes[0, 1].set_ylabel('Accuracy')
        axes[0, 1].set_title('Preference Accuracy Over Time')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
    
    # Plot 3: Reward Margins
    if metrics['reward_margins']:
        axes[1, 0].plot(metrics['steps'][:len(metrics['reward_margins'])], 
                       metrics['reward_margins'], linewidth=2, color='purple')
        axes[1, 0].axhline(y=0, color='r', linestyle='--', label='No preference')
        axes[1, 0].set_xlabel('Training Steps')
        axes[1, 0].set_ylabel('Reward Margin')
        axes[1, 0].set_title('Reward Margin (Chosen - Rejected)')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
    
    # Plot 4: KL Divergence
    if metrics['chosen_kls'] and metrics['rejected_kls']:
        steps_kl = metrics['steps'][:len(metrics['chosen_kls'])]
        axes[1, 1].plot(steps_kl, metrics['chosen_kls'], 
                       linewidth=2, label='Chosen KL', color='blue')
        axes[1, 1].plot(steps_kl, metrics['rejected_kls'], 
                       linewidth=2, label='Rejected KL', color='orange')
        axes[1, 1].set_xlabel('Training Steps')
        axes[1, 1].set_ylabel('KL Divergence')
        axes[1, 1].set_title('KL Divergence from Reference')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n📊 Interpretation:")
    print("  Top-left: Loss should decrease (model learning)")
    print("  Top-right: Accuracy should increase above 0.5 (better than random)")
    print("  Bottom-left: Margin should increase (stronger preferences)")
    print("  Bottom-right: KL should stay reasonable (not diverging from reference)")
else:
    print("⚠️  No training metrics available (may need more logging steps)")

## Part 10: Compare DPO vs IPO Loss

In [ ]:
# Compare DPO and IPO loss on same data
print("Comparing DPO vs IPO loss...\n")

# Get a batch
with torch.no_grad():
    # Compute log probs for first example
    policy_chosen_logp = compute_sequence_log_probs(
        policy_chosen_output.logits,
        chosen_tokens['input_ids'],
        chosen_tokens['attention_mask']
    )
    policy_rejected_logp = compute_sequence_log_probs(
        policy_rejected_output.logits,
        rejected_tokens['input_ids'],
        rejected_tokens['attention_mask']
    )
    ref_chosen_logp = compute_sequence_log_probs(
        ref_chosen_output.logits,
        chosen_tokens['input_ids'],
        chosen_tokens['attention_mask']
    )
    ref_rejected_logp = compute_sequence_log_probs(
        ref_rejected_output.logits,
        rejected_tokens['input_ids'],
        rejected_tokens['attention_mask']
    )
    
    # Compute both losses
    dpo_loss_val, dpo_details = dpo_loss(
        policy_chosen_logp, policy_rejected_logp,
        ref_chosen_logp, ref_rejected_logp,
        beta=0.1, return_details=True
    )
    
    ipo_loss_val, ipo_details = ipo_loss(
        policy_chosen_logp, policy_rejected_logp,
        ref_chosen_logp, ref_rejected_logp,
        beta=0.1, return_details=True
    )

print("="*80)
print("LOSS COMPARISON")
print("="*80)
print(f"\nDPO Loss:  {dpo_loss_val.item():.4f}")
print(f"  Formula: -log(σ(β * logits))")
print(f"  Accuracy: {dpo_details['accuracy']:.2%}")

print(f"\nIPO Loss:  {ipo_loss_val.item():.4f}")
print(f"  Formula: (β * logits - 0.5)²")
print(f"  Accuracy: {ipo_details['accuracy']:.2%}")

print("\n💡 Key Differences:")
print("  • DPO uses log-sigmoid loss (classification-like)")
print("  • IPO uses squared loss (regression-like)")
print("  • IPO more robust to noisy preferences and outliers")
print("  • DPO typically performs slightly better on clean data")
print("="*80)

## Part 11: Generate Text with Trained Model

Let's see how the trained model responds to prompts.

In [ ]:
def generate_comparison(prompt, policy_model, reference_model, max_length=50):
    """Generate text from both policy and reference model for comparison."""
    tokenizer = policy_model.tokenizer
    
    # Tokenize prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    # Generate from policy (trained)
    with torch.no_grad():
        policy_output = policy_model.generate(
            inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_new_tokens=max_length,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    
    # Generate from reference (baseline)
    with torch.no_grad():
        ref_output = reference_model.generate(
            inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_new_tokens=max_length,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    
    # Decode
    policy_text = tokenizer.decode(policy_output[0], skip_special_tokens=True)
    ref_text = tokenizer.decode(ref_output[0], skip_special_tokens=True)
    
    return policy_text, ref_text

# Test prompts
test_prompts = [
    "What is the capital of France?",
    "Explain machine learning in simple terms:",
    "How do I make coffee?",
]

print("="*80)
print("GENERATION COMPARISON: DPO-Trained vs Reference")
print("="*80)

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n{'='*80}")
    print(f"Example {i}")
    print(f"{'='*80}")
    print(f"\n📝 Prompt: {prompt}")
    
    policy_text, ref_text = generate_comparison(prompt, policy_model, reference_model)
    
    print(f"\n🎓 DPO-Trained Model:\n{policy_text}")
    print(f"\n📚 Reference Model:\n{ref_text}")
    print(f"\n💡 Notice: DPO model should produce more helpful, preferred responses")

print(f"\n{'='*80}")

## Part 12: Hyperparameter Tuning Guide

### Key Hyperparameters for DPO

#### 1. Beta (β) - Most Important!

**What it does:** Controls KL penalty (how far policy can deviate from reference)

**Range:** 0.05 - 0.5

**Effects:**
- **Low (0.05-0.1)**: Aggressive learning, strong preferences, risk of overfitting
- **Medium (0.1-0.2)**: Balanced (recommended starting point)
- **High (0.3-0.5)**: Conservative, stay close to reference, slower learning

**When to adjust:**
- Training unstable → Increase beta
- Not learning enough → Decrease beta
- KL divergence too high → Increase beta

#### 2. Learning Rate

**DPO:** 5e-7 (much lower than SFT!)

**Why so low?** DPO is sensitive; high LR causes instability

**When to adjust:**
- Loss not decreasing → Try 1e-6
- Training unstable → Try 1e-7

#### 3. Number of Epochs

**Typical:** 1 epoch

**Why?** DPO learns quickly, overfits easily

**When to adjust:**
- Accuracy still increasing → Try 2 epochs
- Overfitting (train >> eval) → Reduce to 1 epoch

#### 4. Batch Size

**Per-device:** 2-4 (small due to pairs)

**Effective (with accumulation):** 16-32 (larger is better)

**How:** Increase `gradient_accumulation_steps` not batch size

### Example Configs for Different Scenarios

In [ ]:
# Scenario 1: Conservative (safe, stable)
conservative_config = {
    "beta": 0.3,
    "learning_rate": 1e-7,
    "num_epochs": 1,
    "use_case": "First time, want stability",
}

# Scenario 2: Balanced (recommended starting point)
balanced_config = {
    "beta": 0.1,
    "learning_rate": 5e-7,
    "num_epochs": 1,
    "use_case": "Default, works well for most cases",
}

# Scenario 3: Aggressive (fast learning, higher risk)
aggressive_config = {
    "beta": 0.05,
    "learning_rate": 1e-6,
    "num_epochs": 2,
    "use_case": "Model not learning enough",
}

# Scenario 4: Noisy data (use IPO)
noisy_data_config = {
    "beta": 0.2,
    "learning_rate": 5e-7,
    "num_epochs": 1,
    "loss_type": "ipo",  # More robust
    "use_case": "Preferences are noisy or inconsistent",
}

print("DPO HYPERPARAMETER GUIDE")
print("="*80)

for name, config in [
    ("Conservative", conservative_config),
    ("Balanced", balanced_config),
    ("Aggressive", aggressive_config),
    ("Noisy Data", noisy_data_config),
]:
    print(f"\n{name} Config:")
    for key, value in config.items():
        print(f"  {key}: {value}")

print("\n" + "="*80)

## Part 13: DPO vs Traditional RLHF - When to Use Which?

### Use DPO When:

✅ **Starting from scratch**
- No existing reward model
- Want simplest approach
- Limited compute resources

✅ **Have good preference data**
- Clear chosen vs rejected examples
- Sufficient quantity (1000+)
- Relatively clean labels

✅ **Want stability**
- Don't want to deal with RL instability
- Need predictable training
- Production deployment priority

✅ **Resource constrained**
- Limited GPU memory
- Can't run 4 models simultaneously
- Want faster training

### Use Traditional RLHF (PPO) When:

⚠️ **Have trained reward model**
- Already invested in reward model
- Reward model performs well
- Can leverage for other tasks

⚠️ **Need exploration**
- Want model to discover novel strategies
- Preference data limited
- Open-ended creative tasks

⚠️ **Online learning**
- Adapt during training
- Interactive scenarios
- Reward model can provide signal for new data

⚠️ **Complex reward functions**
- Safety constraints
- Task-specific metrics
- Multiple objectives

### Practical Comparison Table

In [ ]:
import pandas as pd

comparison_data = {
    'Aspect': [
        'Stages',
        'Models',
        'Training Type',
        'Complexity',
        'Stability',
        'Speed',
        'Memory',
        'Hyperparameters',
        'Performance',
    ],
    'DPO': [
        '2 (SFT → DPO)',
        '2 (Policy, Reference)',
        'Supervised',
        'Simple',
        'Very Stable',
        'Fast',
        'Medium',
        'Few (3-4)',
        'Similar to PPO',
    ],
    'PPO (RLHF)': [
        '3 (SFT → Reward → PPO)',
        '4 (Policy, Critic, Reference, Reward)',
        'Reinforcement Learning',
        'Complex',
        'Can Diverge',
        'Slow',
        'High',
        'Many (10+)',
        'Benchmark',
    ],
}

df = pd.DataFrame(comparison_data)
print("\nDPO vs PPO-RLHF Comparison:")
print(df.to_string(index=False))

print("\n💡 Bottom Line:")
print("  • DPO: Simpler, more stable, easier to use → Recommended for most cases")
print("  • PPO: More complex, but more flexible → Use when you need exploration or have reward model")

## Part 14: Summary and Key Takeaways

### What We Learned

1. **DPO Simplifies RLHF**
   - No separate reward model needed
   - No complex RL required
   - Direct supervised learning from preferences

2. **DPO Loss Function**
   - Compares log probabilities (policy vs reference)
   - Creates implicit reward: $r = \beta \log(\pi/\pi_{\text{ref}})$
   - Optimizes to prefer chosen over rejected

3. **Two Models Required**
   - Policy (trainable): Learns from preferences
   - Reference (frozen): Provides KL constraint

4. **Key Hyperparameters**
   - Beta: Controls KL penalty (0.1 default)
   - Learning rate: Much lower than SFT (5e-7)
   - Epochs: Usually just 1

5. **When to Use DPO**
   - Starting from scratch
   - Want simplicity and stability
   - Have good preference data
   - Resource constrained

### Next Steps

1. **Try on Real Data**
   - Use Anthropic HH-RLHF dataset
   - Scale up to 10K+ examples
   - Train on GPU for faster results

2. **Experiment with Hyperparameters**
   - Test different beta values (0.05, 0.1, 0.2)
   - Try IPO loss for noisy data
   - Tune learning rate if needed

3. **Compare with PPO**
   - Implement full RLHF pipeline
   - Compare results and complexity
   - Understand trade-offs

4. **Deploy and Iterate**
   - Use trained model in production
   - Collect new preference data
   - Retrain periodically

### Resources

**Papers:**
- [DPO Paper](https://arxiv.org/abs/2305.18290): Rafailov et al., 2023
- [IPO Paper](https://arxiv.org/abs/2310.12036): Azar et al., 2023

**Documentation:**
- `docs/DPO_THEORY.md`: Mathematical deep dive
- `docs/DPO_CONFIGURATION.md`: Hyperparameter guide
- `src/core/dpo/`: Implementation code

**Training Scripts:**
- `scripts/train/train_dpo.py`: Production training
- `configs/experiment/dpo_gpt2_synthetic.yaml`: Quick test config
- `configs/experiment/dpo_gpt2_hh_rlhf.yaml`: Real data config

### Congratulations! 🎉

You now understand:
- ✅ Why DPO is simpler than PPO-RLHF
- ✅ How DPO loss works mathematically
- ✅ How to train models with DPO
- ✅ When to use DPO vs traditional RLHF
- ✅ How to tune hyperparameters

**Ready to train production models!** 🚀

## Appendix: Quick Reference

### Train DPO on Real Data (Command Line)

```bash
# Quick test (synthetic)
python scripts/train/train_dpo.py \
    experiment=dpo_gpt2_synthetic \
    device=cuda \
    training.fp16=true

# Real training (Anthropic HH-RLHF)
python scripts/train/train_dpo.py \
    experiment=dpo_gpt2_hh_rlhf \
    device=cuda \
    training.fp16=true

# Custom hyperparameters
python scripts/train/train_dpo.py \
    technique.beta=0.2 \
    training.learning_rate=1e-6 \
    training.num_epochs=2
```

### Google Colab

```python
# Clone and setup
!git clone https://github.com/ars137th/llm-post-training.git
%cd llm-post-training
!pip install -e ".[gpu]"

# Train
!python scripts/train/train_dpo.py \
    experiment=dpo_gpt2_synthetic \
    device=cuda \
    training.fp16=true
```

### Key Code Snippets

```python
# Load models
policy = LanguageModel.from_pretrained("gpt2", use_lora=True)
reference = LanguageModel.from_pretrained("gpt2")

# Compute DPO loss
from src.core.dpo.loss import dpo_loss
loss = dpo_loss(
    policy_chosen_logps,
    policy_rejected_logps,
    ref_chosen_logps,
    ref_rejected_logps,
    beta=0.1,
)

# Train with DPOTrainer
trainer = DPOTrainer(
    model=policy.model,
    ref_model=reference.model,
    args=training_args,
    train_dataset=train_dataset,
    beta=0.1,
)
trainer.train()
```